In [1]:
import redis.asyncio as redis  
import asyncio
import nest_asyncio
nest_asyncio.apply()

from datetime import datetime

from alpaca.data.live.stock import StockDataStream
import os 

stock_stream = StockDataStream(os.environ['API_KEY'], os.environ['SECRET_KEY'])
tickers = ['AAPL','AMD','CSCO','GOOG','INTC','JNPR','META','MSFT','NFLX','NVDA','TSLA']

redis_client = redis.Redis(host='localhost', port=6379, decode_responses=True)

async def push_ohlc_data(bar):
    bar = {k: v for k, v in bar}
    bar['timestamp'] = bar['timestamp'].isoformat()
    # Add the new OHLC tick to the Redis Stream
    await redis_client.xadd(f"alpaca_{bar['symbol']}", bar)

    # Trim the stream to keep only the last 30 ticks
    await redis_client.xtrim(f"ohlc_stream:{bar['symbol']}", maxlen=100)
    print(f"Pushed OHLC Tick: {bar}")


In [ ]:
stock_stream.subscribe_bars(push_ohlc_data, *tickers)
stock_stream.run()

Pushed OHLC Tick: {'symbol': 'AAPL', 'timestamp': '2025-03-20T13:04:00+00:00', 'open': 214.2, 'high': 214.25, 'low': 214.2, 'close': 214.25, 'volume': 500.0, 'trade_count': 6.0, 'vwap': 214.22}
Pushed OHLC Tick: {'symbol': 'AAPL', 'timestamp': '2025-03-20T13:05:00+00:00', 'open': 214.04, 'high': 214.04, 'low': 214.04, 'close': 214.04, 'volume': 201.0, 'trade_count': 4.0, 'vwap': 214.04}
